# Tekne dedektörü v6 -- güncel 5185 görüntülük veri, imgsz=960, freeze YOK

`yolov8s.pt`'den sıfırdan başlıyor, backbone dondurulmamış (v4_s ile adil karşılaştırma), imgsz=960 (pipeline'ın inference çözünürlüğüyle eşleşiyor), güncel `boat_v4s_frozen_v2_bundle.zip` (3944 train + 1241 val = 5185 görüntü) kullanılıyor.

**Önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir` ile GPU seç (A100 önerilir), ve `boat_v4s_frozen_v2_bundle.zip`'i Google Drive'ına yükle.

In [ ]:
# 1) GPU kontrol
!nvidia-smi

In [ ]:
# 2) Drive bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip'i aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/trainFreeze/boat_v4s_frozen_v2_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml icindeki Mac path'ini Colab'a gore duzelt
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) Sifirdan egitim -- yolov8s.pt'den, freeze YOK, imgsz=960, guncel 5185 goruntuluk veri
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=960,
    device=0,
    batch=32,
    patience=40,
    project='/content/work/runs_boat_yolo',
    name='boat_v6_960_5k',
)

In [ ]:
# 7) Egitim Colab oturumu koptugu icin durduysa, yukaridaki hucreleri (Drive baglama,
# unzip, pip install, path duzeltme) tekrar calistirdiktan sonra bunu (6. hucre yerine) calistir:
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v6_960_5k/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 8) Bitince (veya ara sonuclari kaybetmemek icin ara sira) Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v6_960_5k_results
!cp -r /content/work/runs_boat_yolo/boat_v6_960_5k /content/drive/MyDrive/boat_v6_960_5k_results/
print('Kopyalandi: Google Drive > boat_v6_960_5k_results > boat_v6_960_5k')

## Bitince Mac'e geri alma

Drive > `boat_v6_960_5k_results` altındaki `boat_v6_960_5k` klasörünü indir, içindeki `weights/best.pt` ve `weights/last.pt`'yi Mac'inde `depth-anything/runs/detect/runs_boat_yolo/boat_v6_960_5k/weights/` altına koy (klasör yoksa oluştur).